# ==========================================
# Enerji Perakende Veri Analizi
## NOTEBOOK 03 — Veri Hikayesi ve İş Önerileri
# ==========================================

# Notebook 03 – Veri Hikayesi ve Stratejik Analiz

## Problem Tanımı

Enerji perakende şirketi, ilçeler arası tüketim farklılıklarını,
müşteri segmentlerini ve tahsilat risklerini anlamak istemektedir.

Temel sorular:

- İlçeler arası tüketim farkının nedeni nedir?
- Yüksek tüketimli aboneler ortalamayı nasıl etkiliyor?
- Hangi müşteri segmentleri tahsilat riski oluşturmaktadır?
- Mevsimsel etki tüketim farkında belirleyici midir?

## Hipotezler

H1: İlçeler arası ortalama tüketim farkı müşteri profilinden kaynaklanmaktadır.

H2: Yüksek tüketimli az sayıdaki abone genel ortalamayı yukarı çekmektedir.

H3: Düşük tüketimli müşteriler daha yüksek tahsilat riski taşımaktadır.

H4: Mevsimsel desen tüm ilçelerde benzer olup,
genel tüketim farkının ana nedeni değildir.

In [ ]:
import pandas as pd
import numpy as np

from data_loader import load_data

df_tahsilat, df_tahsilat_1, df_tahakkuk, df_tahakkuk_1, df_tahakkuk_2 = load_data(verbose=False)

df_all = pd.concat([df_tahakkuk, df_tahakkuk_1, df_tahakkuk_2], ignore_index=True)

print("Veriler hazır.")

In [ ]:
import os
import matplotlib.pyplot as plt

OUTPUT_DIR = "../outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def save_fig(name):
    plt.savefig(f"{OUTPUT_DIR}/{name}.png",
                dpi=300,
                bbox_inches="tight")

print("Output klasörü hazır:", OUTPUT_DIR)

# İlçeler Arası Tüketim ve Tahsilat Analizi

Bu analizde Amasya iline bağlı Hamamözü, Gümüşhacıköy ve Göynücek ilçelerinin
elektrik tüketim ve ödeme davranışları karşılaştırılmıştır.

Amaç:
- İlçeler arası tüketim farklarını açıklamak
- Hesap sınıfı dağılımının etkisini incelemek
- Geç ödeme davranışını analiz etmek
- Riskli müşteri segmentlerini belirlemek
- Tahsilat süreçlerine veri destekli öneriler geliştirmek

## Hipotezler

H1: Gümüşhacıköy ve Göynücek ilçelerinde ortalama tüketim Hamamözü’nden yüksektir.

H2: Ticari ve tarımsal hesap oranı yüksek olan ilçelerde ortalama kWh daha fazladır.

H3: Yüksek tüketim segmentinde geç ödeme oranı daha yüksektir.

H4: Mevsimsel artış Temmuz–Ağustos döneminde zirve yapmaktadır.

In [ ]:
df_all["Mali yıl/dönem"] = pd.to_datetime(df_all["mali_yil_donem"])

df_all["Ay"] = df_all["Mali yıl/dönem"].dt.month

In [ ]:
aylik_ilce = df_all.groupby(["ilce", "Ay"])["kwh"].mean().reset_index()

aylik_ilce.head()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))

for ilce in aylik_ilce["ilce"].unique():
    data = aylik_ilce[aylik_ilce["ilce"] == ilce]
    plt.plot(data["Ay"], data["kwh"], label=ilce)

plt.legend()
plt.xlabel("Ay")
plt.ylabel("Ortalama kWh")
plt.title("İlçe Bazlı Aylık Ortalama Tüketim")

plt.tight_layout()

plt.savefig(f"{OUTPUT_DIR}/ilce_bazli_aylik_tuketim.png",
            dpi=300,
            bbox_inches="tight")

plt.show()

In [ ]:
df_all.groupby("ilce")["kwh"].agg(["mean", "median", "std", "count"]).sort_values("mean", ascending=False)

In [ ]:
df_all.groupby("ilce")["kwh"].agg(
    Ortalama="mean",
    Medyan="median",
    Std="std",
    Min="min",
    Max="max",
    Kayit_Sayisi="count"
).sort_values("Ortalama", ascending=False)

In [ ]:
pd.crosstab(
    df_all["ilce"],
    df_all["Hesap Sınıfı"],
    normalize="index"
) * 100

In [ ]:
df_mesken = df_all[df_all["Hesap Sınıfı"] == "Mesken"]

df_mesken.groupby("ilce")["kwh"].mean()

### İlçe Karşılaştırma Analizi

Genel ortalama tüketim değerleri ilçeler arasında farklılık göstermektedir. 
Ancak medyan değerlerin birbirine yakın olması, farkın büyük tüketicilerden 
kaynaklandığını göstermektedir.

Mesken aboneleri ayrı analiz edildiğinde ilçeler arasındaki fark önemli ölçüde 
azalmaktadır. Bu durum, tüketim farklılıklarının temel olarak ticari ve yüksek 
kapasiteli abonelerden kaynaklandığını göstermektedir.

Hamamözü ilçesinin düşük ortalama tüketim göstermesinin temel nedeni, 
yüksek tüketimli abonelerin daha az olmasıdır.

In [ ]:
ilce_hesap_oran = pd.crosstab(
    df_all["ilce"],
    df_all["Hesap Sınıfı"],
    normalize="index"
) * 100

ilce_hesap_oran.round(2)

In [ ]:
buyuk_aboneler = df_all[
    df_all["Hesap Sınıfı"].isin([
        "Sanayi",
        "Karayolları Aydınlatma",
        "Arıtma Tesisleri",
        "İçme-Kullanma Suyu (Belediye)",
        "Tarımsal (Kooperatif)"
    ])
]

buyuk_aboneler.groupby("ilce")["kwh"].mean()

### Büyük Tüketici Analizi

İlçeler arası ortalama tüketim farkının temel nedeni, yüksek kapasiteli 
ticari ve kamu aboneleridir. Mesken tüketimleri ilçeler arasında benzer 
seviyedeyken, büyük tüketicilerin varlığı genel ortalamayı ciddi şekilde 
etkilemektedir.

Özellikle Göynücek ilçesinde çok yüksek tüketimli aboneler bulunmaktadır. 
Bu durum, ilçenin genel ortalamasını yukarı çekmektedir.

Sonuç olarak, ortalama değerler tek başına bölgesel karşılaştırma için 
yeterli değildir. Medyan değerler daha sağlıklı bir gösterge sunmaktadır.

In [ ]:
df_all["kwh"].describe().round(2)

In [ ]:
df_all["Segment"] = pd.qcut(
    df_all["kwh"],
    q=4,
    labels=["Low", "Medium", "High", "Very High"]
)

df_all["Segment"].value_counts()

In [ ]:
df_all.groupby("Segment", observed=False)["kwh"].agg(["mean", "median", "count"])

In [ ]:
pd.crosstab(
    df_all["ilce"],
    df_all["Segment"],
    normalize="index"
) * 100

### Segment Bazlı Tüketim Analizi

Tüketim dağılımı dört segmente ayrılmıştır: Low, Medium, High ve Very High.

Low segment düşük tüketimli meskenleri temsil ederken,
Very High segment az sayıda yüksek kapasiteli aboneleri içermektedir.

Very High segmentinde ortalama tüketim, medyan değerin oldukça üzerindedir.
Bu durum, enerji gelirinin önemli bir kısmının az sayıdaki yüksek tüketiciden
geldiğini göstermektedir.

İlçeler bazında incelendiğinde, Gümüşhacıköy ve Göynücek ilçelerinde
yüksek tüketimli abonelerin etkisi daha belirgindir.
Hamamözü ilçesi ise daha düşük ve homojen bir tüketim yapısına sahiptir.

Bu durum, tahsilat riskinin ve gelir yoğunlaşmasının segment bazlı
yönetilmesi gerektiğini göstermektedir.

In [ ]:
df_segment = df_all[["sozlesme_hesap_no", "Segment"]].drop_duplicates()

In [ ]:
df_odeme = df_tahsilat_1.merge(
    df_segment,
    left_on="Söz.hsp.(bağımsız)",
    right_on="sozlesme_hesap_no",
    how="left"
)

df_odeme.head()

In [ ]:
gec_sutunlar = [
    col for col in df_tahsilat_1.columns
    if "Son Ödeme (" in col
]

gec_sutunlar

In [ ]:
df_segment = df_all[["sozlesme_hesap_no", "Segment"]].drop_duplicates()

df_odeme = df_tahsilat_1.merge(
    df_segment,
    left_on="Söz.hsp.(bağımsız)",
    right_on="sozlesme_hesap_no",
    how="left"
)

df_odeme["Gec_Odeme"] = df_odeme[gec_sutunlar].notna().any(axis=1)

In [ ]:
segment_risk = pd.crosstab(
    df_odeme["Segment"],
    df_odeme["Gec_Odeme"],
    normalize="index"
) * 100

segment_risk.round(2)

### Segment Bazlı Geç Ödeme Analizi

Geç ödeme oranları incelendiğinde, düşük tüketimli (Low segment) 
abonelerin daha yüksek risk taşıdığı görülmektedir (%28.6).

Tüketim arttıkça geç ödeme oranı azalmaktadır. 
Very High segment en düşük risk oranına sahiptir (%25.9).

Bu durum, yüksek tüketimli abonelerin ödeme disiplininin 
daha güçlü olduğunu göstermektedir.

Tahsilat süreçleri özellikle Low segment üzerinde 
yoğunlaştırılmalıdır.

In [ ]:
df_sorted = df_all.sort_values("kwh", ascending=False)

top_5 = int(len(df_sorted) * 0.05)

top_5_oran = (
    df_sorted.head(top_5)["kwh"].sum()
    / df_sorted["kwh"].sum()
) * 100

top_5_oran

### Tüketim Yoğunlaşması Analizi

Toplam tüketimin yaklaşık %50’si, en yüksek tüketim yapan %5 müşteri
grubundan gelmektedir.

Bu durum, tüketim ve potansiyel gelir yapısının az sayıdaki büyük
abonede yoğunlaştığını göstermektedir.

Yoğunlaşma oranının yüksek olması, operasyonel ve finansal açıdan
bağımlılık riskini artırmaktadır. Büyük abonelere yönelik özel
müşteri yönetimi ve risk takibi stratejileri geliştirilmelidir.

In [ ]:
def gini(array):
    array = np.array(array)
    array = array[array >= 0]  # negatifleri çıkar (varsa)
    
    array = np.sort(array)
    n = len(array)
    
    cumulative = np.cumsum(array)
    gini_index = (n + 1 - 2 * np.sum(cumulative) / cumulative[-1]) / n
    
    return gini_index

gini_kwh = gini(df_all["kwh"])

gini_kwh

### Tüketim Eşitsizliği (Gini Analizi)

Tüketim dağılımı için hesaplanan Gini katsayısı 0.67 olarak bulunmuştur.

Bu değer, tüketimin yüksek derecede eşitsiz dağıldığını göstermektedir.
Az sayıdaki büyük tüketici, toplam tüketimin önemli bir kısmını
oluşturmaktadır.

Bu durum, gelir yapısının yoğunlaşma riski taşıdığını ve büyük
abonelere yönelik stratejik müşteri yönetiminin kritik olduğunu
göstermektedir.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Negatifleri çıkar
kwh_values = df_all["kwh"]
kwh_values = kwh_values[kwh_values >= 0]

# Küçükten büyüğe sırala
sorted_kwh = np.sort(kwh_values)

# Kümülatif toplam
cum_kwh = np.cumsum(sorted_kwh)

# Normalize et
cum_kwh = cum_kwh / cum_kwh[-1]

# Nüfus yüzdesi
cum_pop = np.arange(1, len(sorted_kwh) + 1) / len(sorted_kwh)

# Grafik
plt.figure(figsize=(6,6))
plt.plot(cum_pop, cum_kwh, label="Lorenz Eğrisi")
plt.plot([0,1], [0,1], linestyle="--", label="Tam Eşitlik")

plt.xlabel("Müşteri Oranı")
plt.ylabel("Kümülatif Tüketim Oranı")
plt.title("Lorenz Eğrisi – Tüketim Eşitsizliği")
plt.legend()
plt.tight_layout()

# 🔥 PNG olarak kaydet
plt.savefig(f"{OUTPUT_DIR}/lorenz_egrisi_tuketim_esitsizligi.png",
            dpi=300,
            bbox_inches="tight")

plt.show()

### Lorenz Eğrisi Yorumu – Tüketim Eşitsizliği

Lorenz eğrisi incelendiğinde, tüketim dağılımının tam eşitlik doğrusundan
belirgin şekilde saptığı görülmektedir. Eğrinin alt bölgede uzun süre yatay
seyretmesi, müşteri kitlesinin büyük bir kısmının toplam tüketimin oldukça
küçük bir bölümünü oluşturduğunu göstermektedir.

Grafikten görüldüğü üzere:

- İlk %50 müşteri toplam tüketimin yaklaşık %10–15’ini gerçekleştirmektedir.
- Tüketimin önemli bir kısmı son dilimde yoğunlaşmaktadır.
- En yüksek tüketim yapan küçük bir müşteri grubu sistemin büyük bölümünü taşımaktadır.

Bu yapı, hesaplanan Gini katsayısı (0.67) ile tutarlıdır ve yüksek derecede
tüketim eşitsizliğini doğrulamaktadır.

Sonuç olarak, enerji tüketimi homojen dağılmamaktadır. Gelir ve operasyonel
yük önemli ölçüde az sayıdaki büyük abonede yoğunlaşmıştır. Bu durum,
büyük müşterilere yönelik özel risk yönetimi ve müşteri ilişkileri
stratejilerinin kritik olduğunu göstermektedir.

## Hipotezlerin Değerlendirilmesi

H1: İlçeler arası ortalama tüketim farkı vardır → Doğrulandı.  
H2: Fark büyük tüketicilerden kaynaklanmaktadır → Doğrulandı.  
H3: Yüksek tüketim segmenti daha risklidir → Reddedildi.  
H4: Tüketim eşitsizdir → Doğrulandı (Gini = 0.67).

# Notebook_03 – Genel Değerlendirme ve Stratejik Sonuçlar

Bu çalışma kapsamında enerji tüketim verileri; ilçe bazlı yapısal farklılıklar, müşteri segmentasyonu, tahsilat davranışı ve tüketim eşitsizliği perspektiflerinden çok boyutlu olarak analiz edilmiştir. Analiz süreci yalnızca tanımlayıcı istatistiklerle sınırlı kalmamış; dağılım yapıları, risk desenleri ve operasyonel etkiler birlikte değerlendirilmiştir.

---

## 1. İlçe Bazlı Yapısal Farklılıklar

İlçe karşılaştırmaları, ortalama tüketim değerleri arasında belirgin farklar olduğunu göstermektedir. Ancak medyan tüketim değerlerinin birbirine oldukça yakın olması, bu farkın tüm müşteri kitlesine homojen şekilde dağılmadığını ortaya koymaktadır.

Bu durum, ortalamayı yukarı çeken az sayıdaki yüksek tüketimli abonelerin varlığına işaret etmektedir. Başka bir deyişle:

- İlçeler arasındaki fark geniş tabanlı bir tüketim kültüründen değil,
- Yüksek hacimli sınırlı sayıdaki aboneden kaynaklanmaktadır.

Mevsimsel analiz sonuçları tüm ilçelerde benzer bir desen göstermektedir. Yaz aylarında yükselen, kış aylarında normalize olan tüketim davranışı ilçeler arasında büyük bir yapısal ayrışma göstermemektedir. Bu bulgu, tüketim farklarının temel nedeninin iklimsel değil, müşteri kompozisyonu ve hesap sınıfı dağılımı olduğunu düşündürmektedir.

---

## 2. Müşteri Segmentasyonu ve Dağılım Yapısı

Tüketim dağılımının sağa çarpık olduğu gözlemlenmiştir. Bu, sistemde küçük tüketimli çok sayıda müşteri ve az sayıda yüksek tüketimli müşteri bulunduğunu göstermektedir.

Segmentasyon analizi sonucunda:

- Düşük tüketimli segment geniş bir müşteri kitlesini temsil etmektedir.
- Very High segment müşteri sayısı olarak küçük olmakla birlikte toplam tüketim içinde orantısız derecede yüksek paya sahiptir.

Bu yapı, klasik 80/20 prensibine benzer bir yoğunlaşmayı işaret etmektedir. Gelirin ve operasyonel yükün önemli bir bölümü az sayıdaki büyük abonede toplanmaktadır.

---

## 3. Tahsilat Riski ve Davranışsal Desenler

Tahsilat analizi, beklenenin aksine yüksek tüketimli segmentten ziyade düşük tüketimli segmentte daha yüksek ödeme riski bulunduğunu göstermiştir.

Bu durum şu şekilde yorumlanabilir:

- Düşük tüketimli aboneler gelir seviyesi açısından daha kırılgan olabilir.
- Küçük tutarlı faturalar ödeme önceliğinde geri planda kalabilir.
- Davranışsal olarak düşük tüketimli segment daha düzensiz ödeme eğilimi gösterebilir.

Bu bulgu, tahsilat stratejisinin yalnızca fatura tutarına göre değil, segment bazlı risk profili üzerinden tasarlanması gerektiğini ortaya koymaktadır.

---

## 4. Tüketim Eşitsizliği ve Konsantrasyon Analizi

Gini katsayısının 0.67 olarak hesaplanması, tüketimde oldukça yüksek bir eşitsizlik olduğunu göstermektedir. Lorenz eğrisi analizine göre toplam tüketimin yaklaşık yarısı en yüksek %5’lik müşteri grubundan gelmektedir.

Bu yapı şirket açısından iki önemli anlam taşımaktadır:

1. Gelir Konsantrasyonu Riski:  
   Büyük abonelerin kaybı veya ödeme gecikmesi ciddi nakit akışı riski yaratabilir.

2. Operasyonel Önceliklendirme:  
   Büyük tüketiciler için özel müşteri yönetimi, birebir takip ve kurumsal hizmet modeli geliştirilmesi gerekebilir.

---

## 5. Stratejik İş Önerileri

### Segment Bazlı Yönetim Modeli
- Düşük tüketimli ve yüksek riskli segment için erken uyarı sistemi
- Otomatik ödeme teşvik kampanyaları
- Mikro taksitlendirme modelleri

### Büyük Tüketiciler İçin Özel Yönetim
- Key account yaklaşımı
- Proaktif tahsilat takibi
- Özel sözleşme ve fiyatlandırma optimizasyonu

### Veri Tabanlı Tahsilat Skoru
- Tüketim seviyesi + ödeme gecikmesi + mevsimsellik kombinasyonundan risk skoru üretimi
- Tahsilat ekiplerinin önceliklendirilmesi

### Mevsimsel Talep Yönetimi
- Yaz aylarında pik yapan segmentler için kampanya ve kapasite planlaması
- Tarımsal ve ticari aboneler için dönemsel destek programları

---

## 6. Genel Sonuç

Bu analiz, enerji tüketiminin homojen ve dengeli bir yapı sergilemediğini; aksine yoğunlaşmış, segment bazlı ve davranışsal farklılıklar içeren karmaşık bir sistem olduğunu göstermektedir.

Ortalama değerlere dayalı değerlendirmeler tüketim gerçekliğini tam olarak yansıtmamaktadır. Dağılım yapısının incelenmesi, eşitsizlik analizi ve segment bazlı yaklaşım şirket için daha yüksek karar kalitesi sağlamaktadır.

Enerji perakende yönetimi yalnızca toplam tüketim veya toplam tahsilat rakamları üzerinden değil; segment bazlı risk, eşitsizlik ve davranış analitiği temelinde kurgulanmalıdır.

Bu çalışma, veri odaklı ve stratejik karar alma kültürünün operasyonel verimlilik ve finansal sürdürülebilirlik açısından kritik önem taşıdığını ortaya koymaktadır.